# Qwen-Image 2.1

- **支持显卡**：A100 (80GB)、H100 (80GB)、RTX 6000 (96GB)
- **默认配置**：BF16（主模型 / 文本编码器 / VAE）、2K 分辨率（可切换 1K / 原图）、40 步、单张
- **访问方式**：通过 Colab Secrets 自动读取 `CF_TUNNEL_TOKEN`，挂载 Cloudflare Tunnel 直连自有域名与 API
- **执行顺序**：`1. 环境检测` → `2. 安装 ComfyUI` → `3. 下载模型与工作流` → `4. 启动并打开 ComfyUI`

| 工作流 | 功能 | 输入要求 | LoRA |
| --- | --- | --- | --- |
| `01_Qwen_Image_2_1_T2I` | 文生图 | 提示词 | 无需 |
| `02_Qwen_Image_2_1_Reference` | 单图编辑 | 上传 1 张图片 + `<image1>` 指令 | 无需 |
| `03_Qwen_Image_2_1_T2I_LoRA` | LoRA 文生图 | 动漫提示词 | 动漫一致性 LoRA |
| `04_Qwen_Image_2_1_Reference_LoRA` | LoRA 单图编辑 | 上传 1 张图片 + `<image1>` 指令 | 动漫一致性 LoRA |
| `05_Qwen_Image_2_1_Two_Images` | 双图编辑 | 上传 2 张图片 + `<image1>` / `<image2>` 指令 | 无需 |


### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、PyTorch 与 CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`，显存 80GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。


In [ ]:
# @title 1. 环境检测 {display-mode: "form"}
import subprocess, torch

print(subprocess.run(
    ["nvidia-smi"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
).stdout)

if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 CUDA GPU。")

gpu = torch.cuda.get_device_name(0)
memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("显卡：", gpu)
print(f"显存： {memory:.1f} GiB")
print("PyTorch 版本：", torch.__version__)
print("CUDA 版本：", torch.version.cuda)

supported_gpu = any(k in gpu.upper() for k in ("A100", "H100", "6000"))
if not supported_gpu or memory < 70 or not torch.cuda.is_bf16_supported():
    raise RuntimeError("本模型需要 80GB 级显卡（A100 80GB / H100 80GB / RTX 6000 96GB，显存 >= 70 GiB），请切换 runtime。")

print(f"[OK] GPU 环境正常：{gpu} ({memory:.1f} GiB)")


### 2. 安装 ComfyUI
- **功能**：安装指定版本官方 ComfyUI 至 `/content/ComfyUI` 并配置依赖与 `cloudflared`。
- **说明**：自动跳过 `torch / torchvision / torchaudio / xformers`，保留 Colab 原生 PyTorch 环境。
- **使用**：直接运行，输出 `[OK] ComfyUI 安装完成` 后继续。


In [ ]:
# @title 2. 安装 ComfyUI {display-mode: "form"}
import re, subprocess, sys, urllib.request
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
COMFY_REVISION = 'c9d8a6e69c4b5ab7fa0f789e7b988c172cd31fc9'

def run(args):
    subprocess.run(args, cwd=COMFY_DIR if COMFY_DIR.exists() else None, check=True)

if COMFY_DIR.is_symlink():
    raise RuntimeError("ComfyUI 目录是软链接，请使用新的 runtime。")
fresh = not COMFY_DIR.exists()
if fresh:
    subprocess.run(["git", "clone", "--depth", "1", "--no-checkout", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY_DIR)], check=True)
if not (COMFY_DIR / ".git").is_dir():
    raise RuntimeError("ComfyUI 目录不是 Git 仓库，请使用新的 runtime。")
origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=COMFY_DIR, text=True).strip()
if origin.rstrip("/").removesuffix(".git") != "https://github.com/Comfy-Org/ComfyUI":
    raise RuntimeError("现有 ComfyUI 不是官方仓库，请使用新的 runtime。")
current = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=COMFY_DIR, text=True).strip()
if not fresh:
    changes = subprocess.check_output(["git", "status", "--porcelain"], cwd=COMFY_DIR, text=True).strip()
    if changes:
        raise RuntimeError("现有 ComfyUI 有未提交修改，请使用新的 runtime。")
if fresh or current != COMFY_REVISION:
    run(["git", "fetch", "--depth", "1", "origin", COMFY_REVISION])
    run(["git", "checkout", "--detach", COMFY_REVISION])

skip = {"torch", "torchvision", "torchaudio", "xformers"}
lines = []
for line in (COMFY_DIR / "requirements.txt").read_text().splitlines():
    name = re.split(r"[<>=!~\[]", line.strip(), maxsplit=1)[0].strip().lower()
    if name not in skip:
        lines.append(line)
requirements = Path("/content/qwen_comfy_requirements.txt")
requirements.write_text("\n".join(lines) + "\n")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), "huggingface_hub", "hf_xet"])

cloudflared_bin = Path("/usr/local/bin/cloudflared")
if not cloudflared_bin.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(cloudflared_bin),
    )
    cloudflared_bin.chmod(0o755)

print("ComfyUI:", COMFY_REVISION[:12], "| cloudflared: 已就绪")
print("[OK] ComfyUI 安装完成。")


### 3. 下载模型与工作流
- **功能**：下载 Qwen-Image 2.1 BF16 权重（约 32.4 GB）与默认动漫 LoRA 至 `/content/ComfyUI/models`，同步 GitHub 工作流至 `user/default/workflows`。
- **参数**：
  - `下载默认动漫一致性_LoRA`：下载 `03 / 04` 所需的 `Qwen2.1_Anime_consistency.safetensors`（约 168 MB，默认开启）。
  - `WORKFLOW_REF`：工作流 Git 分支或 Commit（默认 `main`）。
- **使用**：直接运行；已存在且校验通过的模型自动跳过，同名工作流直接覆盖。


In [ ]:
# @title 3. 下载模型与工作流 {display-mode: "form"}
下载默认动漫一致性_LoRA = True # @param {type:"boolean"}
WORKFLOW_REF = "main" # @param {type:"string"}

import hashlib, json, os, shutil, tempfile, urllib.error, urllib.request
from pathlib import Path
from urllib.parse import quote, urlencode
from huggingface_hub import hf_hub_download

COMFY_DIR = Path("/content/ComfyUI")
MODEL_REPO = 'Comfy-Org/Qwen-Image-2.1'
MODEL_REVISION = 'cb504a4090723e43f17ad01cec0359490e2de613'
MODEL_FILES = [
    {'file': 'diffusion_models/qwen_image_2.1_bf16.safetensors', 'size': 14230280616},
    {'file': 'text_encoders/qwen3vl_8b_bf16.safetensors', 'size': 17534334616},
    {'file': 'vae/qwen_image_2.1_vae_bf16.safetensors', 'size': 675509688},
]
MODEL_DIR = COMFY_DIR / "models"

def local_directory(path):
    path = Path(path)
    for parent in [path, *path.parents]:
        if parent == COMFY_DIR.parent:
            break
        if parent.is_symlink():
            raise RuntimeError(f"目录是软链接，请使用新的 runtime：{parent}")
    path.mkdir(parents=True, exist_ok=True)

for path in [MODEL_DIR, COMFY_DIR / "input", COMFY_DIR / "output"]:
    local_directory(path)

def download_models(manifest, directory):
    missing = sum(item["size"] for item in manifest if not (directory / item["file"]).is_file() or (directory / item["file"]).stat().st_size != item["size"])
    if missing and shutil.disk_usage(directory).free < missing + 2 * 1024**3:
        raise RuntimeError("本地磁盘空间不足；Qwen 2.1 模型合计约 32.4 GB。")
    for item in manifest:
        target = directory / item["file"]
        local_directory(target.parent)
        if target.is_symlink():
            raise RuntimeError(f"模型文件是软链接：{target}")
        force = target.is_file() and target.stat().st_size != item["size"]
        result = Path(hf_hub_download(repo_id=MODEL_REPO, filename=item["file"], revision=MODEL_REVISION, local_dir=str(directory), force_download=force))
        if result.resolve() != target.resolve() or not target.is_file() or target.stat().st_size != item["size"]:
            raise RuntimeError(f"模型下载不完整或路径不匹配：{target}")
        print(target.name)

download_models(MODEL_FILES, MODEL_DIR)

WORKFLOW_REPO = "boltguo/colab"
WORKFLOW_FOLDER = "workflows/qwen-image-2.1"
WORKFLOW_DIR = COMFY_DIR / "user/default/workflows"
LORA_DIR = MODEL_DIR / "loras"
LORA_REPO = "WarmBloodAban/Qwen-Image-2.1-LoRAs"
LORA_REVISION = "c4ab5473bfdf585fc19cfd1e280f79d2b0c79947"
LORA_FILE = "Qwen2.1_Anime_consistency.safetensors"
LORA_SIZE = 167830408
LORA_SHA256 = "0c171eb802ea8051b511f2d93c1743eeadd030316255aa98fa65d40809366752"
EXPECTED_LOADERS = {
    "UNETLoader": "qwen_image_2.1_bf16.safetensors",
    "CLIPLoader": "qwen3vl_8b_bf16.safetensors",
    "VAELoader": "qwen_image_2.1_vae_bf16.safetensors",
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

def download_default_lora():
    local_directory(LORA_DIR)
    target = LORA_DIR / LORA_FILE
    if target.is_symlink():
        raise ValueError("LoRA 文件是软链接，未写入。")
    if target.exists():
        if not target.is_file() or target.stat().st_size != LORA_SIZE or sha256_file(target) != LORA_SHA256:
            raise ValueError("同名 LoRA 与默认版本不同；保留原文件，请先检查或另存。")
        print("默认 LoRA 已存在：", target)
        return
    if shutil.disk_usage(LORA_DIR).free < LORA_SIZE + 1024**3:
        raise RuntimeError("磁盘空间不足，未下载 LoRA。")
    result = Path(hf_hub_download(repo_id=LORA_REPO, filename=LORA_FILE,
                                 revision=LORA_REVISION, local_dir=str(LORA_DIR)))
    if result.resolve() != target.resolve() or target.is_symlink() or target.stat().st_size != LORA_SIZE or sha256_file(target) != LORA_SHA256:
        raise RuntimeError("LoRA 下载路径、大小或 SHA256 校验失败。")
    print("默认 LoRA 已下载：", target)

def validate_workflow_models(data):
    if not isinstance(data, dict) or not isinstance(data.get("nodes"), list) or not data["nodes"] or not isinstance(data.get("links"), list):
        raise ValueError("工作流 JSON 缺少 nodes 或 links。")
    for node in data["nodes"]:
        if not isinstance(node, dict) or not isinstance(node.get("type"), str):
            raise ValueError("工作流节点结构无效。")
        expected = EXPECTED_LOADERS.get(node["type"])
        values = node.get("widgets_values") or []
        if expected and (not isinstance(values, list) or not values or values[0] != expected):
            raise ValueError(f"{node['type']} 的模型文件名与本 notebook 下载映射不一致。")
    return data

def report_lora_requirements(name, data):
    for node in data["nodes"]:
        if node.get("type") != "LoraLoaderModelOnly" or node.get("mode", 0) == 4:
            continue
        values = node.get("widgets_values") or []
        filename = values[0] if isinstance(values, list) and values else None
        safe_name = isinstance(filename, str) and bool(filename) and Path(filename).name == filename and "\\" not in filename
        target = LORA_DIR / filename if safe_name else None
        if target is None or not target.is_file() or target.is_symlink():
            print(f"前置要求：{name} 需准备 {LORA_DIR / str(filename)}。")

def github_json(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Qwen-Colab-Workflow-Downloader", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        body = response.read(2_000_001)
        if len(body) > 2_000_000 or "text/html" in response.headers.get("Content-Type", "").lower():
            raise ValueError("GitHub 返回网页或文件超过 2 MB，未保存。")
    try:
        return json.loads(body)
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise ValueError("GitHub 返回内容不是 JSON；未保存。") from error

def install_workflow(name, data):
    if not isinstance(name, str) or not name.lower().endswith(".json") or Path(name).name != name or "\\" in name:
        raise ValueError("工作流文件名无效。")
    validate_workflow_models(data)
    local_directory(WORKFLOW_DIR)
    target = WORKFLOW_DIR / name
    if target.is_symlink() or (target.exists() and not target.is_file()):
        raise ValueError("目标工作流是软链接或目录，未写入。")
    fd, temporary = tempfile.mkstemp(prefix=".qwen-workflow-", dir=WORKFLOW_DIR)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as file:
            json.dump(data, file, ensure_ascii=False, indent=2)
            file.write("\n")
        os.replace(temporary, target)
        print("工作流已下载：", name)
        report_lora_requirements(name, data)
        return True
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

def report_github_failure(error, name=None):
    label = f"工作流 {name}" if name else "GitHub 工作流目录"
    if isinstance(error, urllib.error.HTTPError) and error.code == 404:
        print(f"{label} 返回 404：可能尚未推送到 GitHub，或 WORKFLOW_REF={WORKFLOW_REF!r} 不存在。推送后重跑第 3 格；保留当前工作流。")
    else:
        print(f"{label} 下载失败：{error}。未保存错误页面，保留当前工作流。")

def download_github_workflows():
    ref = WORKFLOW_REF.strip()
    if not ref:
        raise ValueError("WORKFLOW_REF 不能为空。")
    api = f"https://api.github.com/repos/{WORKFLOW_REPO}/contents/{WORKFLOW_FOLDER}?" + urlencode({"ref": ref})
    try:
        entries = github_json(api)
        if not isinstance(entries, list):
            raise ValueError("GitHub 目录响应不是文件列表。")
    except (urllib.error.URLError, OSError, ValueError) as error:
        report_github_failure(error)
        return
    entries = [entry for entry in entries if isinstance(entry, dict) and entry.get("type") == "file" and str(entry.get("name", "")).lower().endswith(".json")]
    if not entries:
        print("该 GitHub 目录目前没有 JSON 工作流；推送后重跑第 3 格。")
    for entry in sorted(entries, key=lambda entry: entry["name"]):
        name = entry["name"]
        try:
            if not isinstance(name, str) or Path(name).name != name or "\\" in name:
                raise ValueError("GitHub 文件名无效。")
            url = f"https://raw.githubusercontent.com/{WORKFLOW_REPO}/{quote(ref, safe='')}/{WORKFLOW_FOLDER}/{quote(name, safe='')}"
            install_workflow(name, github_json(url))
        except (urllib.error.URLError, OSError, ValueError) as error:
            report_github_failure(error, name)

local_directory(WORKFLOW_DIR)
if 下载默认动漫一致性_LoRA:
    download_default_lora()
download_github_workflows()
print("[OK] 模型与工作流准备完成。")


### 4. 启动并打开 ComfyUI
- **功能**：后台启动 ComfyUI（`127.0.0.1:8188`），自动从 Colab Secrets 读取 `CF_TUNNEL_TOKEN` 启动 Cloudflare Tunnel。
- **要求**：在 Colab 左侧 **🔑 Secrets（密钥）** 中添加 `CF_TUNNEL_TOKEN` 并开启访问权限（保存一次即可，无需每次输入）。
- **使用**：直接运行；启动后直接通过你在 Cloudflare 绑定的域名访问 UI 或调用 API。


In [ ]:
# @title 4. 启动并打开 ComfyUI {display-mode: "form"}

import json, os, re, signal, socket, subprocess, sys, time, urllib.request
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
BASE = "http://127.0.0.1:8188"
LOG = Path("/content/qwen_comfyui.log")

def get_json(route):
    with urllib.request.urlopen(BASE + route, timeout=3) as response:
        return json.load(response)

def ready():
    try:
        get_json("/system_stats")
        return True
    except Exception:
        return False

def get_listen_addrs():
    try:
        with socket.socket(socket.AF_INET6, socket.SOCK_STREAM) as s:
            s.bind(("::1", 0))
        return "127.0.0.1,::1"
    except OSError:
        return "127.0.0.1"

def stop_idle_comfy():
    try:
        q = get_json("/queue")
        if q.get("queue_running") or q.get("queue_pending"):
            return False
    except Exception:
        pass
    subprocess.run(["pkill", "-f", "_comfy_userdata_compat.py"], check=False)
    time.sleep(2)
    return True

CF_LOG = Path("/content/cloudflared.log")
CF_PID = Path("/content/cloudflared.pid")

def resolve_cf_token():
    t = ""
    try:
        from google.colab import userdata
        t = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        t = ""
    if not t:
        t = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
    return t.split()[-1] if t else ""

def start_cf_tunnel(token):
    subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False)
    if CF_PID.exists():
        CF_PID.unlink(missing_ok=True)
    time.sleep(1)
    cloudflared_bin = Path("/usr/local/bin/cloudflared")
    if not cloudflared_bin.exists():
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            str(cloudflared_bin),
        )
        cloudflared_bin.chmod(0o755)
    with CF_LOG.open("w") as cf_log:
        cf_proc = subprocess.Popen(
            [
                str(cloudflared_bin),
                "tunnel",
                "--no-autoupdate",
                "--protocol", "http2",
                "--edge-ip-version", "4",
                "run",
                "--url", "http://127.0.0.1:8188",
                "--token", token,
            ],
            stdin=subprocess.DEVNULL,
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    CF_PID.write_text(str(cf_proc.pid))
    deadline = time.time() + 30
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            print(CF_LOG.read_text(errors="replace")[-4000:])
            raise RuntimeError("Cloudflare Tunnel 启动失败，请检查 CF_TUNNEL_TOKEN 是否正确。")
        log_text = CF_LOG.read_text(errors="replace") if CF_LOG.exists() else ""
        if "Registered tunnel connection" in log_text:
            if "Updated to new configuration" not in log_text:
                time.sleep(1)
                log_text = CF_LOG.read_text(errors="replace") if CF_LOG.exists() else ""
            clean_log = log_text.replace('\"', '"')
            routes = re.findall(r'"hostname"\s*:\s*"([^"]+)"\s*,\s*"service"\s*:\s*"([^"]+)"', clean_log)
            if not routes:
                routes = [(h, s) for s, h in re.findall(r'"service"\s*:\s*"([^"]+)"\s*,\s*"hostname"\s*:\s*"([^"]+)"', clean_log)]
            if routes:
                for host, srv in routes:
                    print(f"[OK] Cloudflare Tunnel 已连接 (HTTP/2): https://{host} -> {srv}")
                    if "8188" not in srv:
                        print(f"[WARN] 检测到域名 {host} 在 Cloudflare 后台转发目标为 {srv}（非 ComfyUI 的 8188 端口）！")
                        print("       请前往 Cloudflare Zero Trust -> Networks -> Tunnels -> Public Hostname，将 Service URL 改为 http://127.0.0.1:8188")
            else:
                print("[OK] Cloudflare Tunnel 已连接 (HTTP/2)（请在 Cloudflare 绑定的域名访问 ComfyUI 或调用 API）。")
            return True
        time.sleep(1)
    print("[WARN] Cloudflare Tunnel 已在后台运行，详看 /content/cloudflared.log。")
    return True

cf_token = resolve_cf_token()
if not cf_token:
    raise RuntimeError("未检测到 CF_TUNNEL_TOKEN，请在 Colab 左侧 🔑 Secrets 中添加 CF_TUNNEL_TOKEN 并开启访问权限。")

LAUNCHER = Path("/content/qwen_comfy_userdata_compat.py")
LAUNCHER_SOURCE = r'''
"""Start pinned ComfyUI with narrow workflow aliases and Cloudflare compatibility."""
import os
import runpy
import sys
from pathlib import Path
from aiohttp import web


def add_workflow_aliases(manager, routes):
    originals = [route for route in routes
                 if getattr(route, "path", None) == "/userdata/{file}"
                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]
    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:
        raise RuntimeError("ComfyUI userdata routes changed; compatibility aliases were not installed.")
    for route in originals:
        async def workflow_alias(request, handler=route.handler):
            file = request.match_info.get("file", "")
            user_root = manager.get_request_user_filepath(request, None, create_dir=False)
            candidate = manager.get_request_user_filepath(request, file, create_dir=False)
            if not user_root or not candidate:
                return web.Response(status=403)
            workflow_directory = os.path.join(user_root, "workflows")
            if os.path.islink(workflow_directory):
                return web.Response(status=403)
            workflow_root = os.path.realpath(workflow_directory)
            try:
                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root
            except ValueError:
                within = False
            if not within or os.path.realpath(candidate) == workflow_root:
                return web.Response(status=403)
            return await handler(request)
        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)


def install_cf_middleware():
    import middleware.cache_middleware as cm
    orig_cache_control = cm.cache_control

    @web.middleware
    async def cf_safe_middleware(request, handler):
        response = await orig_cache_control(request, handler)
        if request.method == "GET" and request.path == "/" and getattr(response, "status", None) == 200:
            import server
            inst = getattr(server.PromptServer, "instance", None)
            web_root = getattr(inst, "web_root", None) if inst else None
            index_file = Path(web_root) / "index.html" if web_root else None
            if index_file and index_file.is_file():
                html = index_file.read_text(encoding="utf-8", errors="replace")
                html = html.replace("<script ", '<script data-cfasync="false" ')
                new_resp = web.Response(text=html, content_type="text/html")
                for k, v in response.headers.items():
                    if k.lower() not in {"content-length", "content-type", "etag", "last-modified"}:
                        new_resp.headers[k] = v
                new_resp.headers["Cache-Control"] = "no-store, no-transform, must-revalidate"
                response = new_resp
        if isinstance(response, web.StreamResponse) and not isinstance(response, web.WebSocketResponse) and not response.prepared:
            if getattr(response, "status", None) == 200 and (
                request.path.startswith("/assets/")
                or request.path.endswith((".woff2", ".woff", ".ttf"))
            ):
                response.headers["Cache-Control"] = "public, max-age=604800, immutable, no-transform"
            else:
                cc = response.headers.get("Cache-Control", "")
                if "no-transform" not in cc:
                    response.headers["Cache-Control"] = f"{cc}, no-transform" if cc else "no-transform"
            origin = request.headers.get("Origin")
            if origin and response.headers.get("Access-Control-Allow-Origin") == "*":
                response.headers["Access-Control-Allow-Origin"] = origin
                response.headers["Vary"] = "Origin"
        return response

    cm.cache_control = cf_safe_middleware


def main():
    comfy_root = Path("/content/ComfyUI")
    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():
        raise RuntimeError("ComfyUI source directory is missing or invalid.")
    os.chdir(comfy_root)
    sys.path.insert(0, str(comfy_root))
    sys.argv[0] = str(comfy_root / "main.py")
    import comfy.options
    comfy.options.enable_args_parsing()
    install_cf_middleware()
    from app.user_manager import UserManager
    original = UserManager.add_routes

    def compatible_routes(self, routes):
        original(self, routes)
        add_workflow_aliases(self, routes)

    UserManager.add_routes = compatible_routes
    runpy.run_path("main.py", run_name="__main__")


if __name__ == "__main__":
    main()
'''
launcher_updated = not LAUNCHER.exists() or LAUNCHER.read_text(errors="replace") != LAUNCHER_SOURCE
LAUNCHER.write_text(LAUNCHER_SOURCE)

if ready() and launcher_updated:
    stop_idle_comfy()

if not ready():
    with LOG.open("w") as log:
        qwen_process = subprocess.Popen([sys.executable, str(LAUNCHER), "--listen", get_listen_addrs(), "--port", "8188", "--output-directory", str(COMFY_DIR / "output"), "--enable-cors-header", "--enable-compress-response-body", "--disable-all-custom-nodes", "--disable-partner-nodes"], cwd=COMFY_DIR, stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    deadline = time.time() + 600
    while time.time() < deadline:
        if qwen_process.poll() is not None:
            print(LOG.read_text(errors="replace")[-8000:])
            raise RuntimeError("ComfyUI 启动失败，请查看上方日志。")
        if ready():
            break
        time.sleep(2)
    else:
        raise TimeoutError("ComfyUI 10 分钟内未就绪。")

registered = get_json("/object_info")
if not {"TextEncodeQwenImage21", "QwenImage21Cache", "CustomCombo", "ComfyMathExpression", "ResolutionSelector"}.issubset(registered):
    raise RuntimeError("缺少 Qwen 或内置尺寸节点，请重跑第 2 格安装固定版本，再运行第 4 格。日志：/content/qwen_comfyui.log。")

start_cf_tunnel(cf_token)
